In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/hauuto/catbreed-id.git"
BRANCH = "main"

WORK_DIR = Path("/kaggle/working/catbreed-id")
OUTPUTS_DIR = Path("/kaggle/working/outputs")
HF_REPO_ID = "catbreed-id/catbreed-experiments"

In [ ]:
import subprocess
import sys

if not WORK_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(WORK_DIR)],
        check=True,
    )
else:
    changes = subprocess.check_output(
        ["git", "status", "--porcelain"],
        cwd=WORK_DIR,
        text=True,
    ).strip()

    if changes:
        raise RuntimeError(
            "Repo có thay đổi chưa commit. Kiểm tra trước khi cập nhật."
        )

    subprocess.run(
        ["git", "switch", BRANCH],
        cwd=WORK_DIR,
        check=True,
    )
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", BRANCH],
        cwd=WORK_DIR,
        check=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "-r", str(WORK_DIR / "requirements.txt"),
    ],
    check=True,
)

GIT_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=WORK_DIR,
    text=True,
).strip()

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print("Code sẵn sàng:", GIT_COMMIT)

In [ ]:
import os

from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

api = HfApi(token=os.environ["HF_TOKEN"])
RUN_OWNER = api.whoami()["name"]

# Xác nhận tài khoản truy cập được kho kết quả.
api.repo_info(repo_id=HF_REPO_ID, repo_type="dataset")

print("Tài khoản:", RUN_OWNER)
print("Kho kết quả:", HF_REPO_ID)
print("Runner sẵn sàng")

In [ ]:
import importlib
import json
from datetime import datetime, timezone
from uuid import uuid4

import torch
import yaml

if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

importlib.invalidate_caches()
from src.training.artifacts import save_stage, sync_results

run_id = (
    datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    + "_smoke_"
    + uuid4().hex[:8]
)
run_dir = OUTPUTS_DIR / run_id
run_dir.mkdir(parents=True, exist_ok=False)

info = {
    "run_id": run_id,
    "owner": RUN_OWNER,
    "git_commit": GIT_COMMIT,
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "Kiểm tra workflow, không huấn luyện",
    "is_smoke_test": True,
    "data_version": None,
    "split_id": None,
    "training_seed": None,
    "status": "running",
}

def write_info():
    (run_dir / "info.json").write_text(
        json.dumps(info, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

write_info()

(run_dir / "config.yaml").write_text(
    yaml.safe_dump({
        "mode": "smoke",
        "stages": ["frozen", "finetuned"],
        "hf_repo_id": HF_REPO_ID,
    }),
    encoding="utf-8",
)

(run_dir / "history.csv").write_text(
    "stage,event\nfrozen,smoke_checkpoint\nfinetuned,smoke_checkpoint\n",
    encoding="utf-8",
)

(run_dir / "requirements_freeze.txt").write_text(
    subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"],
        text=True,
    ),
    encoding="utf-8",
)

sync_status = {}

for stage, value in [("frozen", 1.0), ("finetuned", 2.0)]:
    sync_status[stage] = save_stage(
        run_dir=run_dir,
        stage=stage,
        checkpoint={
            "is_smoke_test": True,
            "tensor": torch.tensor([value]),
        },
        metrics={"is_smoke_test": True, "stage": stage},
        repo_id=HF_REPO_ID,
    )

info["status"] = "completed"
info["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
write_info()

sync_status["final"] = sync_results(run_dir, HF_REPO_ID)

print("\nRun ID:", run_id)
for step, success in sync_status.items():
    print(f"{step}: {'Đã đồng bộ' if success else 'Chưa đồng bộ'}")

if not all(sync_status.values()):
    raise RuntimeError(
        f"Có bước chưa đồng bộ. Giữ thư mục để upload lại: {run_dir}"
    )

print("Workflow thử hoàn thành")